In [98]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [99]:
keras.mixed_precision.set_global_policy('mixed_bfloat16')

In [100]:
import pickle
import tarfile
import datetime
import numpy as np
import pandas as pd
import urllib.request
import sklearn.metrics
import matplotlib.pyplot as plt
import zipfile
from joblib import Parallel, delayed
import albumentations as A

from pathlib import Path
import cv2
from typing import Literal

In [101]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: False


In [102]:
BATCH_SIZE = 256
LOGS_DIR = '../logs'
DATA_DIR = '../data'

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
class SegmentationDataset(keras.utils.PyDataset):

    def __init__(
        self,
        path: Path,
        *,
        batch_size: int = 32,
        augmentations: A.Compose | None = None,
        seed: int | None = None,
        shuffle: bool = True,
    ):
        self.path = path
        self.batch_size = batch_size
        self.augmentations = augmentations
        self.data_amount = len(list(path.iterdir())) // 2
        self.rng = np.random.default_rng(seed)
        self.shuffle = shuffle

        self.data_paths = []
        self.masks_paths = []

        for i in range(1, self.data_amount):
            data = path / f"{i}.png"
            mask = path / f"{i}_outline.png"

            self.data_paths.append(data)
            self.masks_paths.append(mask)

        self.indices = np.arange(len(self.data_paths))

        self.data = np.array([self.safe_read(path) for path in self.data_paths])
        self.masks = np.array(
            [
                cv2.cvtColor(self.safe_read(path), cv2.COLOR_BGR2GRAY)
                for path in self.masks_paths
            ]
        )
        self.on_epoch_end()

    def safe_read(self, path) -> np.ndarray:
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Could not read image at {path}")
        return img

    def __len__(self) -> int:
        return int(np.ceil(len(self.data_paths) / self.batch_size))

    def __getitem__(self, idx) -> tuple[np.ndarray, np.ndarray]:
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.data_paths))
        batch_indices = self.indices[start:end]

        return self.data[batch_indices], self.masks[batch_indices]

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

In [104]:
segmentation_path = Path("../data/segmentation")

In [105]:
train_ds = SegmentationDataset(segmentation_path / "train", batch_size=BATCH_SIZE)

In [106]:
val_ds = SegmentationDataset(segmentation_path / "val", batch_size=BATCH_SIZE)